In [65]:
import pyspark
from pyspark import SparkContext
from pyspark.sql import Row
from pyspark.sql import SQLContext
from pyspark import SparkFiles
from pyspark.ml import Pipeline
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier, OneVsRest
import os
import pandas as pd

In [2]:
#CREATE SPARK CONTEXT
#CREATE SQL CONTEXT

sc =SparkContext()
sqlContext = SQLContext(sc)

25/03/13 14:08:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [69]:
#LOAD IRIS DATA
data_dir="data"
file = os.path.join(data_dir,"iris.csv")
print(file)
panda_df = pd.read_csv(file)
print(panda_df.head())
## On définit un schema pour indiquer le type de chaque feature
schema = StructType([
    StructField("sepal_length", DoubleType()),
    StructField("sepal_width", DoubleType()),
    StructField("petal_length", DoubleType()),
    StructField("petal_width", DoubleType()),
    StructField("variety", StringType())
])
iris_df=sqlContext.createDataFrame(panda_df,schema=schema)
iris_df.printSchema()
iris_df.show(5)

data/iris.csv
   sepal_length  sepal_width  petal_length  petal_width variety
0           5.1          3.5           1.4          0.2  Setosa
1           4.9          3.0           1.4          0.2  Setosa
2           4.7          3.2           1.3          0.2  Setosa
3           4.6          3.1           1.5          0.2  Setosa
4           5.0          3.6           1.4          0.2  Setosa
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- variety: string (nullable = true)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|variety|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3

In [13]:
# Caractéristiques Statistiques du DataFrame
iris_df.describe().show()

+-------+------------------+------------------+------------------+------------------+---------+
|summary|      sepal_length|       sepal_width|      petal_length|       petal_width|  variety|
+-------+------------------+------------------+------------------+------------------+---------+
|  count|               150|               150|               150|               150|      150|
|   mean| 5.843333333333334|3.0573333333333323|3.7579999999999996|1.1993333333333336|     null|
| stddev|0.8280661279778636|0.4358662849366982| 1.765298233259466|0.7622376689603465|     null|
|    min|               4.3|               2.0|               1.0|               0.1|   Setosa|
|    max|               7.9|               4.4|               6.9|               2.5|Virginica|
+-------+------------------+------------------+------------------+------------------+---------+



In [15]:
# Nombre total de variétés dans le dataFrame
count = iris_df.count()
print(f'Le Nombre Total de variétés est : {count}')

Le Nombre Total de variétés est : 150


In [18]:
# Pourcentage de CHaque Variété
iris_df.groupBy("variety").count().show()
print("Le DataFrame est équilibré")

+----------+-----+
|   variety|count|
+----------+-----+
| Virginica|   50|
|    Setosa|   50|
|Versicolor|   50|
+----------+-----+

Le DataFrame est équilibré


In [70]:
#Split into training and testing data
(trainingData, testData) = iris_df.randomSplit([0.8, 0.2])
testData.show()

+------------+-----------+------------+-----------+----------+
|sepal_length|sepal_width|petal_length|petal_width|   variety|
+------------+-----------+------------+-----------+----------+
|         4.4|        2.9|         1.4|        0.2|    Setosa|
|         5.1|        3.8|         1.5|        0.3|    Setosa|
|         5.4|        3.9|         1.3|        0.4|    Setosa|
|         5.0|        3.4|         1.6|        0.4|    Setosa|
|         5.2|        3.5|         1.5|        0.2|    Setosa|
|         5.2|        3.4|         1.4|        0.2|    Setosa|
|         5.2|        4.1|         1.5|        0.1|    Setosa|
|         4.5|        2.3|         1.3|        0.3|    Setosa|
|         5.5|        3.5|         1.3|        0.2|    Setosa|
|         5.0|        3.5|         1.6|        0.6|    Setosa|
|         6.4|        3.2|         4.5|        1.5|Versicolor|
|         5.0|        2.0|         3.5|        1.0|Versicolor|
|         5.2|        2.7|         3.9|        1.4|Vers

In [75]:
# Decision Tree Model
indexer = StringIndexer(inputCol="variety", outputCol="label")
assembler = VectorAssembler(inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"], outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
model_dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")

# Pipeline
pipeline = Pipeline(stages=[indexer, assembler, scaler, model_dt])

# Train the model
dt_model = pipeline.fit(trainingData)

# Make predictions
predictions_dt = dt_model.transform(testData)

predictions_dt.select("prediction","variety","label", "features").show(10)

# create evaluator
evaluator_dt = MulticlassClassificationEvaluator(predictionCol="prediction", \
                        labelCol="label",metricName="accuracy")

#Evaluate accuracy
accuracy_dt = evaluator_dt.evaluate(predictions)    

# Print The Accuracy
print(f'Decision Tree Accuracy : {accuracy_dt*100} %')

#Draw a confusion matrix
predictions_dt.groupBy("label","prediction").count().show()

+----------+-------+-----+-----------------+
|prediction|variety|label|         features|
+----------+-------+-----+-----------------+
|       1.0| Setosa|  1.0|[4.4,2.9,1.4,0.2]|
|       1.0| Setosa|  1.0|[5.1,3.8,1.5,0.3]|
|       1.0| Setosa|  1.0|[5.4,3.9,1.3,0.4]|
|       1.0| Setosa|  1.0|[5.0,3.4,1.6,0.4]|
|       1.0| Setosa|  1.0|[5.2,3.5,1.5,0.2]|
|       1.0| Setosa|  1.0|[5.2,3.4,1.4,0.2]|
|       1.0| Setosa|  1.0|[5.2,4.1,1.5,0.1]|
|       1.0| Setosa|  1.0|[4.5,2.3,1.3,0.3]|
|       1.0| Setosa|  1.0|[5.5,3.5,1.3,0.2]|
|       1.0| Setosa|  1.0|[5.0,3.5,1.6,0.6]|
+----------+-------+-----+-----------------+
only showing top 10 rows

Decision Tree Accuracy : 97.36842105263158 %
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  2.0|       0.0|    2|
|  1.0|       1.0|   10|
|  2.0|       2.0|   11|
|  0.0|       0.0|    7|
|  0.0|       2.0|    2|
+-----+----------+-----+



In [74]:
# Random Forest Classifier
model_rf = RandomForestClassifier(featuresCol="features", labelCol="label")

# Pipeline
pipeline_rf = Pipeline(stages=[indexer, assembler, scaler, model_rf])

# Train the model
rf_model = pipeline_rf.fit(trainingData)

# Make predictions
predictions_rf = rf_model.transform(testData)

predictions_rf.select("prediction","variety","label", "features").show(10)

# create evaluator
evaluator_rf = MulticlassClassificationEvaluator(predictionCol="prediction", \
                        labelCol="label",metricName="accuracy")

#Evaluate accuracy
accuracy_rf = evaluator_rf.evaluate(predictions_rf)    

# Print The Accuracy
print(f'Random Forest Classifier Accuracy : {accuracy_rf*100} %')

#Draw a confusion matrix
predictions.groupBy("label","prediction").count().show()

+----------+-------+-----+-----------------+
|prediction|variety|label|         features|
+----------+-------+-----+-----------------+
|       1.0| Setosa|  1.0|[4.4,2.9,1.4,0.2]|
|       1.0| Setosa|  1.0|[5.1,3.8,1.5,0.3]|
|       1.0| Setosa|  1.0|[5.4,3.9,1.3,0.4]|
|       1.0| Setosa|  1.0|[5.0,3.4,1.6,0.4]|
|       1.0| Setosa|  1.0|[5.2,3.5,1.5,0.2]|
|       1.0| Setosa|  1.0|[5.2,3.4,1.4,0.2]|
|       1.0| Setosa|  1.0|[5.2,4.1,1.5,0.1]|
|       1.0| Setosa|  1.0|[4.5,2.3,1.3,0.3]|
|       1.0| Setosa|  1.0|[5.5,3.5,1.3,0.2]|
|       1.0| Setosa|  1.0|[5.0,3.5,1.6,0.6]|
+----------+-------+-----+-----------------+
only showing top 10 rows

Random Forest Classifier Accuracy : 93.75 %
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  2.0|       0.0|    1|
|  1.0|       1.0|   13|
|  2.0|       2.0|   15|
|  0.0|       0.0|    9|
+-----+----------+-----+



In [73]:
# Gradient Boosting

## transformer le classificateur binaire en une multi-classe classifieur à l’aide d’un classificateur d’arbres

gbt = GBTClassifier(labelCol="label", featuresCol="features")

ovr = OneVsRest(classifier=gbt, labelCol="label")
gbt_ovr_pipeline = Pipeline(stages=[indexer, assembler, ovr])

gbt_ovr_model = gbt_ovr_pipeline.fit(trainingData)

gbt_ovr_predictions = gbt_ovr_model.transform(testData)

gbt_ovr_predictions.select("prediction", "label", "features").show()

gbt_ovr_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")

gbt_over_accuracy = gbt_ovr_evaluator.evaluate(gbt_ovr_predictions)
print("Test accuracy = ", gbt_over_accuracy*100, "%")

+----------+-----+-----------------+
|prediction|label|         features|
+----------+-----+-----------------+
|       1.0|  1.0|[4.4,2.9,1.4,0.2]|
|       1.0|  1.0|[5.1,3.8,1.5,0.3]|
|       1.0|  1.0|[5.4,3.9,1.3,0.4]|
|       1.0|  1.0|[5.0,3.4,1.6,0.4]|
|       1.0|  1.0|[5.2,3.5,1.5,0.2]|
|       1.0|  1.0|[5.2,3.4,1.4,0.2]|
|       1.0|  1.0|[5.2,4.1,1.5,0.1]|
|       1.0|  1.0|[4.5,2.3,1.3,0.3]|
|       1.0|  1.0|[5.5,3.5,1.3,0.2]|
|       1.0|  1.0|[5.0,3.5,1.6,0.6]|
|       2.0|  2.0|[6.4,3.2,4.5,1.5]|
|       2.0|  2.0|[5.0,2.0,3.5,1.0]|
|       2.0|  2.0|[5.2,2.7,3.9,1.4]|
|       2.0|  2.0|[6.0,2.2,4.0,1.0]|
|       2.0|  2.0|[6.6,2.9,4.6,1.3]|
|       2.0|  2.0|[5.8,2.7,4.1,1.0]|
|       2.0|  2.0|[6.1,2.9,4.7,1.4]|
|       2.0|  2.0|[6.3,2.5,4.9,1.5]|
|       2.0|  2.0|[6.8,2.8,4.8,1.4]|
|       2.0|  2.0|[5.5,2.4,3.8,1.1]|
+----------+-----+-----------------+
only showing top 20 rows

Test accuracy =  100.0 %
